# 🏆 Churn Prediction: Master Production Pipeline (V5.6 - HYPER+)
**Features**: EDA + Rich Hyperparameter Tuning + MLflow Metrics.

In [ ]:
# 1. Setup
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub -q

import torch
HAS_GPU = torch.cuda.is_available()

import opendatasets as od
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, classification_report
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Data Loading & Profiling
def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_test_final = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_train, df_val = train_test_split(df_raw, test_size=0.15, random_state=42, stratify=df_raw['churn'])

print(f"✅ Data Stats: Train({len(df_train)}), Val({len(df_val)}), Test({len(df_test_final)})")

In [ ]:
# 3. Training with Rich Hyperparameter Search
def run_hyper_experiment(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_va, y_va = df_va.drop(columns=['churn']), df_va['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    preprocessor.fit(X_tr)
    X_va_t = preprocessor.transform(X_va)
    
    if model_type == "xgboost":
        clf = XGBClassifier(scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(), device='cuda' if HAS_GPU else 'cpu', early_stopping_rounds=10)
        f_params = {"clf__eval_set": [(X_va_t, y_va)], "clf__verbose": False}
        # Rich Tuning for XGBoost
        m_params = {
            'clf__max_depth': [3, 5, 7],
            'clf__learning_rate': [0.01, 0.1, 0.2],
            'clf__n_estimators': [100, 200],
            'clf__subsample': [0.8, 1.0]
        }
    elif model_type == "random_forest":
        clf = RandomForestClassifier(class_weight='balanced', random_state=42)
        f_params = {}
        # Rich Tuning for Random Forest
        m_params = {
            'clf__max_depth': [10, 20, None],
            'clf__n_estimators': [100, 200],
            'clf__min_samples_split': [2, 5]
        }
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        f_params = {}
        # Tuning for Logistic Regression
        m_params = {'clf__C': [0.01, 0.1, 1.0, 10.0], 'clf__solver': ['lbfgs', 'liblinear']}
        
    mlflow.set_experiment("Churn_Hyper_Tuning_V5")
    with mlflow.start_run(run_name=f"NB_TUNED_{model_type.upper()}"):
        pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
        # Using n_iter=5 to explore more params while keeping it reasonable for a demo
        search = RandomizedSearchCV(pipe, m_params, n_iter=5, cv=2, scoring='recall', verbose=1, random_state=42)
        search.fit(X_tr, y_tr, **f_params)
        model = search.best_estimator_
        
        print(f"\n🏆 BEST PARAMS ({model_type.upper()}): {search.best_params_}")
        y_pred = model.predict(X_te)
        print(f"\n📊 Classification Report:\n{classification_report(y_te, y_pred)}")
        
        mlflow.log_params(search.best_params_)
        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}")
        print(f"✅ REGISTERED: {model_type.upper()}")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_hyper_experiment(df_train, df_val, df_test_final, m)